# Assignment 3: Tiny Transformer — Tiny Shakespeare
Next-token prediction with a hand-built Transformer in PyTorch.

## Phase 1 — Setup & Reproducibility

In [1]:
import math
import random
import sys
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [2]:
# Environment detection — works identically locally and on Colab via the VS Code extension
ON_COLAB = "google.colab" in sys.modules

if ON_COLAB:
    import subprocess
    subprocess.run(["pip", "install", "tokenizers", "tqdm", "-q"], check=True)
    BASE_DIR = Path("/content")
    print("Running on Colab")
else:
    BASE_DIR = Path("..")
    print("Running locally")

DATA_DIR = BASE_DIR / "data"
CKPT_DIR = BASE_DIR / "outputs" / "checkpoints"
PLOT_DIR = BASE_DIR / "outputs" / "plots"

DATA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR : {DATA_DIR.resolve()}")
print(f"CKPT_DIR : {CKPT_DIR.resolve()}")

Running on Colab
DATA_DIR : /content/data
CKPT_DIR : /content/outputs/checkpoints


In [3]:
# Central config — all hyperparameters live here
config = {
    "vocab_size":  500,
    "seq_len":      50,
    "d_model":     128,
    "n_heads":       4,
    "n_layers":      2,
    "ffn_dim":     256,
    "dropout":     0.1,
    "lr":         3e-4,
    "weight_decay": 0.01,
    "batch_size":   64,
    "num_epochs":   20,
}

## Phase 2 — Data Preparation & Tokenization

### 2a. Load the Tiny Shakespeare corpus

In [4]:
corpus_path = DATA_DIR / "input.txt"

if not corpus_path.exists():
    URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    print("Downloading Tiny Shakespeare...")
    urllib.request.urlretrieve(URL, corpus_path)

with open(corpus_path) as f:
    text = f.read()

print(f"Corpus length: {len(text):,} characters")
print("Sample:")
print(text[:300])

Corpus length: 1,115,394 characters
Sample:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


### 2b. BPE Tokenization

In [5]:
tokenizer_path = DATA_DIR / "tokenizer.json"

if not tokenizer_path.exists():
    print("Training BPE tokenizer...")
    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(
        vocab_size=config["vocab_size"],
        special_tokens=["[UNK]", "[PAD]"],
    )
    tokenizer.train_from_iterator([text], trainer=trainer)
    tokenizer.save(str(tokenizer_path))
    print(f"Tokenizer saved to {tokenizer_path}")
else:
    tokenizer = Tokenizer.from_file(str(tokenizer_path))
    print(f"Tokenizer loaded from {tokenizer_path}")

actual_vocab_size = tokenizer.get_vocab_size()
print(f"Vocab size: {actual_vocab_size}")
config["vocab_size"] = actual_vocab_size

Training BPE tokenizer...
Tokenizer saved to /content/data/tokenizer.json
Vocab size: 500


### 2c. Encode corpus and create sliding-window sequences

In [6]:
encoding = tokenizer.encode(text)
token_ids = encoding.ids
print(f"Total tokens: {len(token_ids):,}")

SEQ_LEN = config["seq_len"]

inputs  = []
targets = []
for i in range(len(token_ids) - SEQ_LEN):
    inputs.append(token_ids[i : i + SEQ_LEN])
    targets.append(token_ids[i + 1 : i + SEQ_LEN + 1])

inputs  = torch.tensor(inputs,  dtype=torch.long)
targets = torch.tensor(targets, dtype=torch.long)
print(f"inputs shape:  {inputs.shape}")
print(f"targets shape: {targets.shape}")

Total tokens: 447,717
inputs shape:  torch.Size([447667, 50])
targets shape: torch.Size([447667, 50])


### 2d. Train / Val split (80 / 20)

In [7]:
n = len(inputs)
split = int(0.8 * n)

train_inputs,  val_inputs  = inputs[:split],  inputs[split:]
train_targets, val_targets = targets[:split], targets[split:]

print(f"Train sequences: {len(train_inputs):,}")
print(f"Val   sequences: {len(val_inputs):,}")

Train sequences: 358,133
Val   sequences: 89,534


In [8]:
class ShakespeareDataset(Dataset):
    def __init__(self, inputs, targets):
        self.inputs  = inputs
        self.targets = targets

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]


BATCH = config["batch_size"]

train_dataset = ShakespeareDataset(train_inputs, train_targets)
val_dataset   = ShakespeareDataset(val_inputs,   val_targets)

train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH, shuffle=False, drop_last=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")

Train batches: 5595
Val   batches: 1399


### 2e. Checkpoint — verify batch shapes

In [9]:
xb, yb = next(iter(train_loader))
print(f"Input  batch shape: {xb.shape}   (expected: [{BATCH}, {SEQ_LEN}])")
print(f"Target batch shape: {yb.shape}   (expected: [{BATCH}, {SEQ_LEN}])")

sample_tokens = xb[0].tolist()
decoded = tokenizer.decode(sample_tokens)
print(f"\nDecoded sample input:\n{decoded}")

Input  batch shape: torch.Size([64, 50])   (expected: [64, 50])
Target batch shape: torch.Size([64, 50])   (expected: [64, 50])

Decoded sample input:
an un li ck ' d be ar - w he l p That c ar ri es no im pr ess ion like the d am . And am I then a man to be be love d ? O m on str ous fa ul t , to


## Phase 3 — Model Implementation

### 3a. Sinusoidal Positional Encoding

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_seq_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_seq_len, d_model)
        pos = torch.arange(max_seq_len).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]

### 3b. RMSNorm

In [11]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-8):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return self.scale * x / rms

### 3c. Causal Multi-Head Self-Attention

In [ ]:
class CausalMultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor):
        B, T, C = x.shape
        H, d_k = self.n_heads, self.d_k

        Q = self.q_proj(x).view(B, T, H, d_k).transpose(1, 2)
        K = self.k_proj(x).view(B, T, H, d_k).transpose(1, 2)
        V = self.v_proj(x).view(B, T, H, d_k).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)

        causal_mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(causal_mask, float("-inf"))

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.attn_drop(attn_weights)

        out = (attn_weights @ V).transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out), attn_weights.detach()

### 3d. Feed-Forward Network

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, ffn_dim: int, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

### 3e. Transformer Block (pre-norm + residual)

In [14]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, ffn_dim: int, dropout: float = 0.0):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn  = CausalMultiHeadAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn   = FeedForward(d_model, ffn_dim, dropout)

    def forward(self, x: torch.Tensor):
        attn_out, attn_weights = self.attn(self.norm1(x))
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x, attn_weights

### 3f. Full Language Model

In [ ]:
class TinyTransformer(nn.Module):
    def __init__(self, cfg: dict):
        super().__init__()
        V  = cfg["vocab_size"]
        D  = cfg["d_model"]
        H  = cfg["n_heads"]
        F  = cfg["ffn_dim"]
        L  = cfg["n_layers"]
        dr = cfg["dropout"]
        T  = cfg["seq_len"]

        self.token_emb = nn.Embedding(V, D)
        self.pos_enc   = SinusoidalPositionalEncoding(D, max_seq_len=T + 1)
        self.drop      = nn.Dropout(dr)
        self.blocks    = nn.ModuleList(
            [TransformerBlock(D, H, F, dr) for _ in range(L)]
        )
        self.norm_out  = RMSNorm(D)
        self.lm_head   = nn.Linear(D, V, bias=False)
        self.lm_head.weight = self.token_emb.weight

        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor):
        x = self.drop(self.pos_enc(self.token_emb(idx)))

        all_attn_weights = []
        for block in self.blocks:
            x, attn_weights = block(x)
            all_attn_weights.append(attn_weights)

        x = self.norm_out(x)
        logits = self.lm_head(x)
        return logits, all_attn_weights

### 3g. Sanity checks

In [16]:
model = TinyTransformer(config).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

dummy = torch.randint(0, config["vocab_size"], (4, config["seq_len"])).to(DEVICE)
with torch.no_grad():
    logits, attn_weights_list = model(dummy)

print(f"Logits shape:        {logits.shape}")
print(f"Attn weight layers:  {len(attn_weights_list)}")
print(f"NaNs in logits:      {logits.isnan().any().item()}")
print(f"Infs in logits:      {logits.isinf().any().item()}")

w = attn_weights_list[0]
T = config["seq_len"]
upper = torch.triu(w[0, 0], diagonal=1)
print(f"Max future attention weight: {upper.max().item():.2e}  (should be ~0)")

Total parameters: 327,552
Logits shape:        torch.Size([4, 50, 500])
Attn weight layers:  2
NaNs in logits:      False
Infs in logits:      False
Max future attention weight: 0.00e+00  (should be ~0)


## Phase 4 — Training Loop

### 4a. Loss, optimizer, and scheduler

In [17]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["lr"],
    weight_decay=config["weight_decay"],
)

total_steps = config["num_epochs"] * len(train_loader)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

print(f"Total training steps: {total_steps:,}")

Total training steps: 111,900


### 4b. Evaluation helper

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, n_batches = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits, _ = model(xb)
        loss = criterion(logits.view(-1, config["vocab_size"]), yb.view(-1))
        total_loss += loss.item()
        n_batches += 1
    return total_loss / n_batches

### 4c. Training loop

In [19]:
train_losses = []
val_losses   = []
best_val_loss = float("inf")

for epoch in range(config["num_epochs"]):
    model.train()
    epoch_loss, n_batches = 0.0, 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{config['num_epochs']}", leave=False)
    for xb, yb in pbar:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        optimizer.zero_grad()
        logits, _ = model(xb)
        loss = criterion(logits.view(-1, config["vocab_size"]), yb.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()
        n_batches  += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = epoch_loss / n_batches
    avg_val_loss   = evaluate(model, val_loader)
    ppl            = math.exp(avg_val_loss)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    print(f"Epoch {epoch+1:02d} | train_loss={avg_train_loss:.4f} | val_loss={avg_val_loss:.4f} | PPL={ppl:.2f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), CKPT_DIR / "best.pt")
        print(f"           ↳ New best checkpoint saved (val_loss={best_val_loss:.4f})")

print(f"\nTraining complete. Best val PPL: {math.exp(best_val_loss):.2f}")

Epoch 01 | train_loss=4.8039 | val_loss=4.4233 | PPL=83.37
           ↳ New best checkpoint saved (val_loss=4.4233)


Epoch 02 | train_loss=4.0377 | val_loss=4.1980 | PPL=66.55
           ↳ New best checkpoint saved (val_loss=4.1980)


Epoch 03 | train_loss=3.7793 | val_loss=4.1066 | PPL=60.74
           ↳ New best checkpoint saved (val_loss=4.1066)


Epoch 04 | train_loss=3.6388 | val_loss=4.0521 | PPL=57.52
           ↳ New best checkpoint saved (val_loss=4.0521)


Epoch 05 | train_loss=3.5535 | val_loss=4.0221 | PPL=55.82
           ↳ New best checkpoint saved (val_loss=4.0221)


...

Epoch 16 | train_loss=3.3405 | val_loss=3.9648 | PPL=52.71


Epoch 17 | train_loss=3.3378 | val_loss=3.9651 | PPL=52.73


Epoch 18 | train_loss=3.3360 | val_loss=3.9632 | PPL=52.62
           ↳ New best checkpoint saved (val_loss=3.9632)


Epoch 19 | train_loss=3.3348 | val_loss=3.9639 | PPL=52.66


Epoch 20 | train_loss=3.3345 | val_loss=3.9638 | PPL=52.66

Training complete. Best val PPL: 52.62


### 4d. Final results summary

In [22]:
# Download outputs back to local machine when running on Colab
if ON_COLAB:
    from google.colab import files
    files.download(str(CKPT_DIR / "best.pt"))
    print("best.pt download triggered")
else:
    print(f"Checkpoint already at {(CKPT_DIR / 'best.pt').resolve()}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

best.pt download triggered


In [21]:
model.load_state_dict(torch.load(CKPT_DIR / "best.pt", map_location=DEVICE))
final_val_loss = evaluate(model, val_loader)
final_ppl = math.exp(final_val_loss)

print(f"Final val loss : {final_val_loss:.4f}")
print(f"Final val PPL  : {final_ppl:.2f}")
print()
print("Epoch | train_loss | val_loss | PPL")
print("-" * 40)
for i, (tl, vl) in enumerate(zip(train_losses, val_losses), 1):
    print(f"  {i:02d}  |   {tl:.4f}   |  {vl:.4f}  | {math.exp(vl):.2f}")

Final val loss : 3.9632
Final val PPL  : 52.62

Epoch | train_loss | val_loss | PPL
----------------------------------------
  01  |   4.8039   |  4.4233  | 83.37
  02  |   4.0377   |  4.1980  | 66.55
  03  |   3.7793   |  4.1066  | 60.74
  04  |   3.6388   |  4.0521  | 57.52
  05  |   3.5535   |  4.0221  | 55.82
  06  |   3.4971   |  4.0041  | 54.82
  07  |   3.4576   |  3.9989  | 54.54
  08  |   3.4287   |  3.9961  | 54.38
  09  |   3.4062   |  3.9812  | 53.58
  10  |   3.3895   |  3.9727  | 53.13
  11  |   3.3760   |  3.9733  | 53.16
  12  |   3.3647   |  3.9693  | 52.95
  13  |   3.3563   |  3.9651  | 52.72
  14  |   3.3495   |  3.9664  | 52.80
  15  |   3.3441   |  3.9640  | 52.67
  16  |   3.3405   |  3.9648  | 52.71
  17  |   3.3378   |  3.9651  | 52.73
  18  |   3.3360   |  3.9632  | 52.62
  19  |   3.3348   |  3.9639  | 52.66
  20  |   3.3345   |  3.9638  | 52.66


In [26]:
from google.colab import drive
drive.mount("/content/drive")
                                                                                                                                 
import shutil
shutil.copy("/content/outputs/checkpoints/best.pt", "/content/drive/MyDrive/best.pt")                                          
print("Saved to Google Drive") 

Mounted at /content/drive
Saved to Google Drive


In [27]:
import json                                                                                                                    
loss_data = {"train_losses": train_losses, "val_losses": val_losses}
with open("/content/drive/MyDrive/loss_data.json", "w") as f:                                                                  
    json.dump(loss_data, f)                                                                                                    
print("Loss data saved")

Loss data saved


In [28]:
print(f"Epochs completed: {len(train_losses)}")
print(f"Best val loss: {best_val_loss:.4f}")                                                                                   
print(f"Best val PPL: {math.exp(best_val_loss):.2f}")

Epochs completed: 20
Best val loss: 3.9632
Best val PPL: 52.62


## Phase 5 — Visualization & Evaluation

### 5a. Loss curves

In [ ]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

with open(CKPT_DIR.parent / 'loss_data.json') as f:
    loss_data = json.load(f)
train_losses = loss_data['train_losses']
val_losses   = loss_data['val_losses']

epochs = range(1, len(train_losses) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, train_losses, label='Train loss')
plt.plot(epochs, val_losses,   label='Val loss')
plt.xlabel('Epoch')
plt.ylabel('Cross-entropy loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / 'loss_curves.png', dpi=150)
plt.show()
print(f'Saved to {PLOT_DIR / "loss_curves.png"}')
print(f'Best val PPL: {math.exp(min(val_losses)):.2f} (epoch {val_losses.index(min(val_losses))+1})')

### 5b. Attention heatmaps

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Load best checkpoint into a fresh model
viz_model = TinyTransformer(config).to(DEVICE)
viz_model.load_state_dict(torch.load(CKPT_DIR / 'best.pt', map_location=DEVICE))
viz_model.eval()

# Pick 3 samples from the val set
sample_indices = [0, 500, 1000]
samples = val_inputs[sample_indices].to(DEVICE)

with torch.no_grad():
    _, attn_weights_list = viz_model(samples)

for layer_idx, layer_weights in enumerate(attn_weights_list):
    # layer_weights: (B, H, T, T)
    for head_idx in range(config['n_heads']):
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f'Layer {layer_idx+1} — Head {head_idx+1}', fontsize=13)

        for ax, sample_idx, seq in zip(axes, sample_indices, samples):
            tokens = seq.tolist()
            labels = [tokenizer.decode([t]) for t in tokens]
            # Trim long labels
            labels = [l[:6] if len(l) > 6 else l for l in labels]

            w = layer_weights[sample_indices.index(sample_idx), head_idx].cpu().numpy()
            im = ax.imshow(w, cmap='viridis', aspect='auto', vmin=0, vmax=w.max())
            ax.set_title(f'Sample {sample_idx}', fontsize=10)
            ax.set_xlabel('Key position (attended to)')
            ax.set_ylabel('Query position (attending)')
            step = max(1, config['seq_len'] // 10)
            ticks = list(range(0, config['seq_len'], step))
            ax.set_xticks(ticks)
            ax.set_yticks(ticks)
            ax.set_xticklabels([labels[i] for i in ticks], rotation=90, fontsize=7)
            ax.set_yticklabels([labels[i] for i in ticks], fontsize=7)
            plt.colorbar(im, ax=ax, fraction=0.046)

        plt.tight_layout()
        fname = PLOT_DIR / f'attn_layer{layer_idx+1}_head{head_idx+1}.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved {fname}')

### 5c. Perplexity summary

In [ ]:
best_ppl  = math.exp(min(val_losses))
final_ppl = math.exp(val_losses[-1])
print(f'Best  val PPL (epoch {val_losses.index(min(val_losses))+1}): {best_ppl:.2f}')
print(f'Final val PPL (epoch {len(val_losses)}):                     {final_ppl:.2f}')

### 5d. Sample text generation

In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=100, top_k=10):
    model.eval()
    ids = tokenizer.encode(prompt).ids
    ids = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(DEVICE)

    for _ in range(max_new_tokens):
        idx_cond = ids[:, -config['seq_len']:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]  # last token

        # Top-k sampling
        if top_k is not None:
            topk_vals, _ = torch.topk(logits, top_k)
            logits[logits < topk_vals[:, -1:]] = float('-inf')

        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)

    return tokenizer.decode(ids[0].tolist())


prompts = ['ROMEO:', 'To be or not to be']
for prompt in prompts:
    print(f'--- Prompt: "{prompt}" ---')
    print(generate(viz_model, tokenizer, prompt, max_new_tokens=80, top_k=10))
    print()